# Retrieval Quality Predictor — Cross-Encoder Distillation

Train a lightweight GradientBoostingRegressor to predict retrieval quality
using cross-encoder scores as targets. This distills the expensive
cross-encoder into a fast feature-based model for real-time quality
estimation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

from src.embeddings.vector_store import VectorStore
from src.retrieval.reranker import Reranker
from src.retrieval.quality_predictor import RetrievalQualityPredictor

## 1. Load Eval Queries and Initialize Models

In [ ]:
eval_df = pd.read_parquet('../data/processed/eval_queries.parquet')
print(f"Eval queries: {len(eval_df)}")

store = VectorStore()
reranker = Reranker()
predictor = RetrievalQualityPredictor()

# Use up to 500 queries for training data generation
queries = eval_df['question'].tolist()[:500]
print(f"Using {len(queries)} queries for distillation")

## 2. Generate Training Data via Cross-Encoder Distillation

For each query, retrieve 20 chunks and score them with the cross-encoder.
These scores become the regression targets.

In [ ]:
from rank_bm25 import BM25Okapi

# Load chunks for BM25
chunks_df = pd.read_parquet('../data/processed/medical_chunks.parquet')
corpus = chunks_df['text'].tolist()
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

training_features = []
training_targets = []

for i, query in enumerate(queries):
    if i % 50 == 0:
        print(f"Processing query {i}/{len(queries)}...")
    
    # Dense retrieval
    results = store.search(query, top_k=20)
    if not results:
        continue
    
    # Cross-encoder scoring (target)
    reranked = reranker.rerank(query, results, top_n=20)
    
    # BM25 scores for the retrieved chunks
    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    
    for r in reranked:
        doc = r['doc']
        ce_score = r['rerank_score']
        
        # Extract features
        cosine_sim = doc.get('score', 0.0)
        chunk_text = doc.get('text', '')
        chunk_tokens = set(chunk_text.lower().split())
        query_token_set = set(query_tokens)
        token_overlap = len(chunk_tokens & query_token_set) / max(len(query_token_set), 1)
        chunk_length = len(chunk_text)
        qtype = doc.get('qtype', '')
        qtype_match = 1 if qtype and qtype.lower() in query.lower() else 0
        
        # Get BM25 score for this specific chunk
        chunk_id = doc.get('chunk_id', '')
        bm25_idx = chunks_df[chunks_df['chunk_id'] == chunk_id].index
        bm25_score = float(bm25_scores[bm25_idx[0]]) if len(bm25_idx) > 0 else 0.0
        
        training_features.append([cosine_sim, bm25_score, token_overlap, chunk_length, qtype_match])
        training_targets.append(ce_score)

X = np.array(training_features)
y = np.array(training_targets)
print(f"\nTraining data: {X.shape[0]} examples, {X.shape[1]} features")

## 3. Train Quality Predictor

In [ ]:
predictor.train_and_log(X, y, experiment_name='retrieval_quality')
print('Quality predictor trained and logged to MLflow.')

## 4. Predicted vs Actual Quality Scores

In [ ]:
y_pred = predictor.pipeline.predict(X)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y, y_pred, alpha=0.3, s=5)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
ax.set_xlabel('Cross-Encoder Score (Actual)')
ax.set_ylabel('Predicted Quality Score')
ax.set_title('Quality Predictor: Predicted vs Actual')
plt.tight_layout()
plt.show()

from sklearn.metrics import mean_squared_error, r2_score
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_pred)):.4f}")
print(f"R²:   {r2_score(y, y_pred):.4f}")

## 5. SHAP Explanations

In [ ]:
import shap

gbr_model = predictor.pipeline.named_steps['model']
scaler = predictor.pipeline.named_steps['scaler']
X_scaled = scaler.transform(X)

explainer = shap.TreeExplainer(gbr_model)
shap_values = explainer.shap_values(X_scaled)

shap.summary_plot(shap_values, X_scaled, feature_names=predictor.FEATURE_NAMES)

In [ ]:
shap.summary_plot(shap_values, X_scaled, feature_names=predictor.FEATURE_NAMES, plot_type='bar')